In [ ]:
# Cell 1: Install dependencies
!pip install transformers==4.40.0 datasets peft==0.10.0 trl==0.8.6 bitsandbytes==0.41.3 accelerate -q
print('Dependencies installed.')

In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/halludet_phi2'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Output dir: {DRIVE_DIR}')

In [ ]:
# Cell 3: Verify GPU
import torch
print(f'GPU : {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Cell 4: Load dataset
# Upload data/processed/dpo_dataset.json via Files panel (left sidebar) > Upload
DATA_PATH = '/content/dpo_dataset.json'

import json
with open(DATA_PATH) as f:
    raw = json.load(f)

from datasets import Dataset
train_ds = Dataset.from_list(raw['train'])
eval_ds  = Dataset.from_list(raw['val'])
print(f'Train: {len(train_ds):,} | Val: {len(eval_ds):,}')

In [ ]:
# Cell 5: Load Phi-2 with QLoRA
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

MODEL = 'microsoft/phi-2'

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)

print('Loading Phi-2...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL, quantization_config=bnb, device_map='auto', trust_remote_code=True
)
model.config.use_cache = False

tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'left'

lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'dense'],
    bias='none'
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()
print('Phi-2 + LoRA loaded.')

In [ ]:
# Cell 6: Load reference model (frozen base)
print('Loading reference model...')
ref_model = AutoModelForCausalLM.from_pretrained(
    MODEL, quantization_config=bnb, device_map='auto', trust_remote_code=True
)
print('Reference model loaded.')

In [ ]:
# Cell 7: DPO Training
from trl import DPOTrainer, DPOConfig
import glob

checkpoints = sorted(glob.glob(f'{DRIVE_DIR}/checkpoint-*'))
resume_from = checkpoints[-1] if checkpoints else None
if resume_from:
    print(f'Resuming from: {resume_from}')

cfg = DPOConfig(
    output_dir=DRIVE_DIR,
    beta=0.1,
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    fp16=True, bf16=False,
    gradient_checkpointing=True,
    optim='paged_adamw_8bit',
    max_length=512,
    max_prompt_length=256,
    evaluation_strategy='steps', eval_steps=50,
    save_strategy='steps', save_steps=50,
    save_total_limit=3,
    logging_steps=10,
    report_to='none',
    remove_unused_columns=False,
)

trainer = DPOTrainer(
    model=model, ref_model=ref_model, args=cfg,
    train_dataset=train_ds, eval_dataset=eval_ds, tokenizer=tokenizer
)

print('Starting DPO training...')
trainer.train(resume_from_checkpoint=resume_from)

In [ ]:
# Cell 8: Save final model
final_dir = f'{DRIVE_DIR}/final'
os.makedirs(final_dir, exist_ok=True)

trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)

saved_files = os.listdir(final_dir)
print(f'Saved to : {final_dir}')
print(f'Files    : {saved_files}')

In [ ]:
# Cell 9: Quick inference test
test_questions = [
    'When did Albert Einstein win the Nobel Prize?',
    'What is the capital of Australia?',
    'Who wrote Romeo and Juliet?',
]

for q in test_questions:
    prompt = f'Answer accurately: {q}\nAnswer:'
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=60, temperature=0.7,
                              do_sample=True, top_p=0.9)
    answer = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f'Q: {q}')
    print(f'A: {answer.strip()}')
    print()